In [8]:
import torch
import time
import tiktoken
import sys
sys.path.append("..")
from model import GPTModel, generate

cfg = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True
}

tokenizer = tiktoken.get_encoding("gpt2")
prompt = "The meaning of life is"
input_ids = tokenizer.encode(prompt)

def benchmark(model, device, label, num_tokens=50, runs=3):
    model.eval()
    model.to(device)
    idx = torch.tensor([input_ids]).to(device)
    
    # warmup
    with torch.no_grad():
        generate(model, idx, max_new_tokens=5, context_length=1024)
    
    times = []
    for _ in range(runs):
        start = time.time()
        with torch.no_grad():
            out = generate(model, idx, max_new_tokens=num_tokens, context_length=1024)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.time() - start
        times.append(elapsed)
    
    avg = sum(times) / len(times)
    tokens_per_sec = num_tokens / avg
    print(f"{label}: {avg:.2f}s | {tokens_per_sec:.1f} tokens/sec")

# Load weights once
from transformers import GPT2LMHeadModel
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_sd = hf_model.state_dict()

Loading weights: 100%|████████████████████████████████████████████| 148/148 [00:00<00:00, 899.29it/s]


In [9]:
def load_gpt2_weights(model, hf_sd):
    sd = model.state_dict()
    
    sd["tok_emb.weight"] = hf_sd["transformer.wte.weight"]
    sd["pos_emb.weight"] = hf_sd["transformer.wpe.weight"]
    
    for i in range(12):
        hf = f"transformer.h.{i}"
        ours = f"trf_blocks.{i}"
        
        sd[f"{ours}.norm1.scale"] = hf_sd[f"{hf}.ln_1.weight"]
        sd[f"{ours}.norm1.shift"] = hf_sd[f"{hf}.ln_1.bias"]
        sd[f"{ours}.norm2.scale"] = hf_sd[f"{hf}.ln_2.weight"]
        sd[f"{ours}.norm2.shift"] = hf_sd[f"{hf}.ln_2.bias"]
        
        qkv_w = hf_sd[f"{hf}.attn.c_attn.weight"]
        q_w, k_w, v_w = qkv_w.split(768, dim=1)
        sd[f"{ours}.att.W_query.weight"] = q_w.T
        sd[f"{ours}.att.W_key.weight"] = k_w.T
        sd[f"{ours}.att.W_value.weight"] = v_w.T
        
        qkv_b = hf_sd[f"{hf}.attn.c_attn.bias"]
        q_b, k_b, v_b = qkv_b.split(768)
        sd[f"{ours}.att.W_query.bias"] = q_b
        sd[f"{ours}.att.W_key.bias"] = k_b
        sd[f"{ours}.att.W_value.bias"] = v_b
        
        sd[f"{ours}.att.W_o.weight"] = hf_sd[f"{hf}.attn.c_proj.weight"].T
        sd[f"{ours}.att.W_o.bias"] = hf_sd[f"{hf}.attn.c_proj.bias"]
        
        sd[f"{ours}.ff.layers.0.weight"] = hf_sd[f"{hf}.mlp.c_fc.weight"].T
        sd[f"{ours}.ff.layers.0.bias"] = hf_sd[f"{hf}.mlp.c_fc.bias"]
        sd[f"{ours}.ff.layers.2.weight"] = hf_sd[f"{hf}.mlp.c_proj.weight"].T
        sd[f"{ours}.ff.layers.2.bias"] = hf_sd[f"{hf}.mlp.c_proj.bias"]
    
    sd["final_norm.scale"] = hf_sd["transformer.ln_f.weight"]
    sd["final_norm.shift"] = hf_sd["transformer.ln_f.bias"]
    sd["out_head.weight"] = hf_sd["transformer.wte.weight"]
    
    model.load_state_dict(sd)
    return model

In [10]:
# Load weights into model
model = GPTModel(cfg)

# Use the same load function from your pretrained notebook
# If it's in model.py:
model = load_gpt2_weights(model, hf_sd)

# Benchmark GPU
benchmark(model, "cuda", "GPU FP32")

# Benchmark CPU
benchmark(model, "cpu", "CPU FP32")

GPU FP32: 1.52s | 32.9 tokens/sec
CPU FP32: 8.93s | 5.6 tokens/sec


In [11]:
# GPU memory usage
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**2:.0f} MB")

# FP16 benchmark
model_fp16 = GPTModel(cfg)
model_fp16 = load_gpt2_weights(model_fp16, hf_sd)
model_fp16 = model_fp16.half()  # convert to FP16
benchmark(model_fp16, "cuda", "GPU FP16")

GPU memory allocated: 682 MB
GPU memory reserved: 1438 MB
GPU FP16: 1.96s | 25.6 tokens/sec


In [12]:
print("=" * 50)
print("BENCHMARK SUMMARY — GPT-2 124M")
print("=" * 50)
print(f"Hardware: NVIDIA RTX 3050 Laptop (4GB VRAM)")
print(f"Model: GPT-2 124M parameters")
print(f"Task: Generate 50 tokens")
print()
print(f"{'Config':<15} {'Time (s)':<12} {'Tokens/sec':<12}")
print("-" * 39)
print(f"{'GPU FP32':<15} {'1.52':<12} {'32.9':<12}")
print(f"{'GPU FP16':<15} {'1.96':<12} {'25.6':<12}")
print(f"{'CPU FP32':<15} {'8.93':<12} {'5.6':<12}")
print()
print(f"GPU Memory: 682 MB allocated / 1438 MB reserved")
print(f"GPU speedup over CPU: ~5.9x")

BENCHMARK SUMMARY — GPT-2 124M
Hardware: NVIDIA RTX 3050 Laptop (4GB VRAM)
Model: GPT-2 124M parameters
Task: Generate 50 tokens

Config          Time (s)     Tokens/sec  
---------------------------------------
GPU FP32        1.52         32.9        
GPU FP16        1.96         25.6        
CPU FP32        8.93         5.6         

GPU Memory: 682 MB allocated / 1438 MB reserved
GPU speedup over CPU: ~5.9x
